In [2]:
"""Weekly Teaching Dashboard Generator v2.

Builds a chronological Monday-Friday planning dashboard from the resolved
teaching model. The dashboard is intended to be run on Friday for the coming
week, but any date window can be generated explicitly for planning/testing.

Sources of truth:
- shared.teaching_model: resolved dated teaching schedules
- course config assignments: explicit assessment due dates

The generator intentionally does NOT encode how multi-meeting weekly content
is divided across class periods. That belongs in lesson planning (e.g. Notion).
"""

from __future__ import annotations

from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
import re

from shared.teaching_model import build_all_course_terms


OUTPUT_DIR = Path("outputs/weekly_briefs")
SECTIONS_FILE = "data/f2026_sections.csv"
COMING_SOON_DAYS = 14

DAY_NAMES = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]


def as_date(value: Any) -> date | None:
    """Convert supported date-like values to datetime.date."""
    if value is None or value == "":
        return None
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    return pd.to_datetime(value).date()


def next_monday(today: date | None = None) -> date:
    """Return the Monday after `today` (or today if today is Sunday)."""
    today = today or date.today()
    days_ahead = (7 - today.weekday()) % 7
    if days_ahead == 0 and today.weekday() != 6:
        days_ahead = 7
    return today + timedelta(days=days_ahead)


def get_dashboard_window(
    start_date: date | str | None = None,
    end_date: date | str | None = None,
    today: date | None = None,
) -> tuple[date, date]:
    """Return the Monday-Sunday dashboard window.

    Normal Friday use: no arguments -> next Monday through Sunday.
    Planning/testing: pass any date and generate the calendar week containing it.
    """
    if start_date is None:
        start = next_monday(today)
    else:
        requested_date = as_date(start_date)
        start = requested_date - timedelta(days=requested_date.weekday())

    if end_date is None:
        end = start + timedelta(days=6)
    else:
        end = as_date(end_date)

    if end < start:
        raise ValueError("end_date must be on or after start_date")

    return start, end


def course_key(course_term: dict) -> tuple[str, str]:
    return course_term["course_code"], course_term["term"]


def primary_schedule(course_term: dict) -> dict:
    return course_term["schedules"][0]

WEEKDAY_LOOKUP = {
    "Mondays": 0,
    "Tuesdays": 1,
    "Wednesdays": 2,
    "Thursdays": 3,
    "Fridays": 4,
}

def parse_meeting_string(class_time):
    """Convert a display string into individual weekly meeting patterns."""
    if not class_time:
        return []

    parts = re.split(r"\s*&\s*", class_time)

    meetings = []

    for part in parts:
        match = re.match(
            r"(Mondays|Tuesdays|Wednesdays|Thursdays|Fridays)\s+(.+)",
            part.strip()
        )

        if match:
            day_name, time_text = match.groups()

            meetings.append({
                "weekday": WEEKDAY_LOOKUP[day_name],
                "class_time": time_text.strip(),
            })

    return meetings

def time_sort_key(time_text):
    """Return a sortable time for strings like '8:30 AM - 10:30 AM'."""
    if not time_text:
        return datetime.max.time()

    match = re.search(r"(\d{1,2}:\d{2}\s*[AP]M)", time_text)

    if not match:
        return datetime.max.time()

    return datetime.strptime(
        match.group(1).replace(" ", ""),
        "%I:%M%p",
    ).time()


def meeting_time_for_date(schedule, event_date):
    """Return the time for the meeting occurring on this particular weekday."""
    meetings = parse_meeting_string(schedule.get("class_time"))

    for meeting in meetings:
        if meeting["weekday"] == event_date.weekday():
            return meeting["class_time"]

    return schedule.get("class_time")

def collect_events(
    course_terms: Iterable[dict],
    start: date,
    end: date,
) -> list[dict]:
    """Collect dated teaching events for the dashboard window.

    Uses the resolved teaching model as the primary source, then adds any
    regular meeting periods contained in the human-readable class_time string
    that the current teaching model does not yet represent separately.
    """
    events = []

    for course_term in course_terms:
        for schedule in course_term["schedules"]:

            # ---------------------------------
            # 1. Events already resolved by teaching_model
            # ---------------------------------
            resolved_dates = set()

            for item in schedule["weeks"]:
                event_date = item["date"]

                if start <= event_date <= end:
                    resolved_dates.add(event_date)

                    events.append(
                        {
                            **item,
                            "course_code": schedule["course_code"],
                            "course_name": schedule["course_name"],
                            "term": schedule["term"],
                            "section": schedule["section"],
                            "class_time": meeting_time_for_date(
                                schedule,
                                event_date,
                            ),
                            "location": item.get("location")
                            or schedule.get("location"),
                        }
                    )

            # ---------------------------------
            # 2. Add regular meeting periods the
            #    current teaching model does not
            #    generate separately.
            # ---------------------------------
            meeting_patterns = parse_meeting_string(
                schedule.get("class_time")
            )

            # Find instructional content represented by the primary
            # schedule date during this dashboard week.
            week_instruction = next(
                (
                    item
                    for item in schedule["weeks"]
                    if start <= item["date"] <= end
                    and item.get("type") == "instruction"
                ),
                None,
            )

            if week_instruction:
                current = start

                while current <= end:
                    for meeting in meeting_patterns:

                        if (
                            current.weekday() == meeting["weekday"]
                            and current not in resolved_dates
                        ):
                            events.append(
                                {
                                    **week_instruction,
                                    "date": current,
                                    "date_str": current.strftime(
                                        "%A, %B %d, %Y"
                                    ),
                                    "class_weekday": current.weekday(),
                                    "class_time": meeting["class_time"],
                                    "location": schedule.get("location"),
                                    "course_code": schedule["course_code"],
                                    "course_name": schedule["course_name"],
                                    "term": schedule["term"],
                                    "section": schedule["section"],
                                    "synthetic_meeting": True,
                                }
                            )

                    current += timedelta(days=1)

    events.sort(
        key=lambda e: (
            e["date"],
            time_sort_key(e.get("class_time")),
            e["course_code"],
            str(e["section"]),
        )
    )

    return events


def get_assignments(course_term: dict) -> list[dict]:
    """Return course-level assessments from the resolved config."""
    return list(primary_schedule(course_term).get("assignments", []))


def assessments_due_between(
    course_term: dict,
    start: date,
    end: date,
) -> list[dict]:
    due = []
    for assignment in get_assignments(course_term):
        due_date = as_date(assignment.get("due_date"))
        if due_date and start <= due_date <= end:
            due.append({**assignment, "due_date": due_date})
    return sorted(due, key=lambda a: (a["due_date"], a["name"]))


def major_assessments_coming_soon(
    course_term: dict,
    after: date,
    days: int = COMING_SOON_DAYS,
) -> list[dict]:
    """Return graded assessments/exams immediately after the dashboard week."""
    horizon = after + timedelta(days=days)
    upcoming = []

    for assignment in get_assignments(course_term):
        due_date = as_date(assignment.get("due_date"))
        if due_date is None or not (after < due_date <= horizon):
            continue

        weight = assignment.get("weight", 0) or 0
        is_exam = assignment.get("type") == "Exam"
        if weight > 0 or is_exam:
            upcoming.append({**assignment, "due_date": due_date})

    return sorted(upcoming, key=lambda a: (a["due_date"], a["name"]))


def format_date(d: date, include_weekday: bool = False) -> str:
    fmt = "%a %b %-d" if include_weekday else "%b %-d"
    try:
        return d.strftime(fmt)
    except ValueError:  # Windows does not support %-d
        fmt = "%a %b %#d" if include_weekday else "%b %#d"
        return d.strftime(fmt)


def format_week_label(event: dict) -> str:
    week = event.get("week")
    event_type = event.get("type", "instruction")

    if week is not None:
        return f"Week {week}"
    if event_type == "orientation":
        return "Week 0 / Orientation"
    return event.get("topic") or event_type.replace("_", " ").title()


def append_blank_tasks(lines: list[str], heading: str, count: int = 2) -> None:
    lines.append(f"**{heading}**")
    for _ in range(count):
        lines.append("- [ ]")
    lines.append("")


def render_event(lines: list[str], event: dict) -> None:
    time = event.get("class_time") or "Time TBD"
    course = event["course_code"]
    section = event["section"]
    location = event.get("location") or "Location TBD"
    topic = event.get("topic") or ""
    event_type = event.get("type", "instruction")

    lines.append(f"### {time} — {course}, Section {section}")
    lines.append(f"📍 {location}")
    lines.append(f"**{format_week_label(event)}:** {topic}")

    if event_type == "instruction":
        notes = event.get("notes")
        if notes:
            lines.append(f"**Class plan / lecture notes:** [Open in Notion]({notes})")
        else:
            lines.append("**Class plan:** _Plan in Notion_ ")

        lab = event.get("lab")
        if lab:
            lines.append(f"**In-class/project focus:** {lab}")
    elif event_type in {"no_class", "holiday"}:
        lines.append("**No regular class.**")

    lines.append("")


def unique_course_events(events: list[dict], course_code: str) -> list[dict]:
    """Deduplicate repeated section rows when building course-wide briefs."""
    seen = set()
    result = []
    for event in events:
        if event["course_code"] != course_code:
            continue
        key = (
            event.get("week"),
            event.get("type"),
            event.get("topic"),
            tuple(event.get("readings") or []),
            event.get("lab"),
        )
        if key not in seen:
            seen.add(key)
            result.append(event)
    return result


def join_value(value: Any) -> str:
    if not value:
        return "None"
    if isinstance(value, list):
        return ", ".join(str(v) for v in value)
    return str(value)


def render_announcement_brief(
    lines: list[str],
    course_term: dict,
    week_events: list[dict],
    start: date,
    end: date,
) -> None:
    master = primary_schedule(course_term)
    code = master["course_code"]
    name = master["course_name"]
    events = unique_course_events(week_events, code)

    lines.append(f"## {code} — {name}")

    course_meetings = [e for e in week_events if e["course_code"] == code]
    if course_meetings:
        meeting_text = "; ".join(
            f"Section {e['section']}: {format_date(e['date'], include_weekday=True)}, "
            f"{e.get('class_time') or 'Time TBD'} ({e.get('location') or 'Location TBD'})"
            for e in course_meetings
            if e.get("type") not in {"no_class", "holiday"}
        )
        if meeting_text:
            lines.append(f"**Meetings this week:** {meeting_text}")
    else:
        lines.append("**Meetings this week:** None")

    instructional = [e for e in events if e.get("type", "instruction") == "instruction"]
    special = [e for e in events if e.get("type", "instruction") != "instruction"]

    if instructional:
        topics = []
        readings = []
        activities = []
        for event in instructional:
            topic = event.get("topic")
            if topic and topic not in topics:
                topics.append(topic)
            for reading in event.get("readings") or []:
                if reading not in readings:
                    readings.append(reading)
            lab = event.get("lab")
            if lab and lab not in activities:
                activities.append(lab)

        lines.append(f"**This week:** {', '.join(topics) if topics else '—'}")
        lines.append(f"**Readings:** {', '.join(readings) if readings else 'None'}")
        if activities:
            lines.append(f"**In class/project focus:** {'; '.join(activities)}")
    else:
        lines.append("**This week:** No regular instructional content scheduled.")
        lines.append("**Readings:** None")

    due = assessments_due_between(course_term, start, end)
    if due:
        due_text = "; ".join(
            f"{a['name']} — {format_date(a['due_date'], include_weekday=True)}"
            for a in due
        )
        lines.append(f"**Due this week:** {due_text}")
    else:
        lines.append("**Due this week:** None")

    upcoming = major_assessments_coming_soon(course_term, end)
    if upcoming:
        upcoming_text = "; ".join(
            f"{a['name']} — {format_date(a['due_date'], include_weekday=True)}"
            for a in upcoming
        )
        lines.append(f"**Coming soon:** {upcoming_text}")
    else:
        lines.append("**Coming soon:** None within the next 14 days")

    if special:
        special_text = "; ".join(
            f"{format_date(e['date'], include_weekday=True)}: {e.get('topic', e.get('type', 'Special event'))}"
            for e in special
        )
        lines.append(f"**Schedule notes:** {special_text}")
    else:
        lines.append("**Schedule notes:** None")

    lines.append("")
    lines.append("**Live updates to add:**")
    lines.append("_Feedback/grading status, project progress, something students should bring, changes since last class, or anything else not contained in the course data._")
    lines.append("")


def build_weekly_dashboard(
    course_terms: list[dict],
    start: date,
    end: date,
) -> str:
    events = collect_events(course_terms, start, end)
    events_by_date = defaultdict(list)
    for event in events:
        events_by_date[event["date"]].append(event)

    lines = [
        "# Weekly Teaching Dashboard",
        f"## {format_date(start, include_weekday=True)} – {format_date(end, include_weekday=True)}",
        "",
    ]

    # Main planning view: Monday-Friday in chronological order.
    monday = start - timedelta(days=start.weekday())
    for offset, day_name in enumerate(DAY_NAMES):
        day = monday + timedelta(days=offset)
        lines.append(f"## {day_name} — {format_date(day)}")
        lines.append("")

        day_events = events_by_date.get(day, [])

        append_blank_tasks(lines, "Before teaching / start of day", 2)

        if day_events:
            for i, event in enumerate(day_events):
                render_event(lines, event)
                if i < len(day_events) - 1:
                    append_blank_tasks(lines, "Between classes", 1)
        else:
            if day_name == "Friday":
                lines.append("**Friday focus / how I want to use today:**")
                lines.append("- [ ]")
                lines.append("- [ ]")
                lines.append("- [ ]")
                lines.append("")
            else:
                lines.append("_No scheduled teaching._")
                lines.append("")

        append_blank_tasks(
            lines,
            "After teaching / end of day" if day_events else "End of day / carry forward",
            2,
        )
        lines.append("---")
        lines.append("")

    # Weekly deadlines / look-ahead.
    lines.append("# Deadlines & Things to Watch")
    lines.append("")

    lines.append("## Due this week")
    any_due = False
    for course_term in course_terms:
        due = assessments_due_between(course_term, start, end)
        if not due:
            continue
        any_due = True
        code = primary_schedule(course_term)["course_code"]
        for assignment in due:
            lines.append(
                f"- **{format_date(assignment['due_date'], include_weekday=True)}** — "
                f"{code}: {assignment['name']}"
            )
    if not any_due:
        lines.append("- None")
    lines.append("")

    lines.append("## Coming soon")
    any_upcoming = False
    for course_term in course_terms:
        upcoming = major_assessments_coming_soon(course_term, end)
        if not upcoming:
            continue
        any_upcoming = True
        code = primary_schedule(course_term)["course_code"]
        for assignment in upcoming:
            lines.append(
                f"- **{format_date(assignment['due_date'], include_weekday=True)}** — "
                f"{code}: {assignment['name']} ({assignment.get('weight', 0)}%)"
            )
    if not any_upcoming:
        lines.append("- None within the next 14 days")
    lines.append("")

    special_events = [e for e in events if e.get("type", "instruction") != "instruction"]
    lines.append("## Schedule notes")
    if special_events:
        for event in special_events:
            lines.append(
                f"- **{format_date(event['date'], include_weekday=True)}** — "
                f"{event['course_code']} Section {event['section']}: {event.get('topic', event.get('type'))}"
            )
    else:
        lines.append("- None")
    lines.append("")

    # Course-wide inputs for Monday announcements.
    lines.append("# Monday Announcement Briefs")
    lines.append("")
    for course_term in course_terms:
        render_announcement_brief(lines, course_term, events, start, end)

    return "\n".join(lines).rstrip() + "\n"


def generate_weekly_dashboard(
    start_date: date | str | None = None,
    end_date: date | str | None = None,
    sections_file: str = SECTIONS_FILE,
    output_dir: Path | str = OUTPUT_DIR,
    today: date | None = None,
) -> Path:
    """Generate and save a weekly dashboard Markdown file."""
    start, end = get_dashboard_window(start_date, end_date, today=today)
    course_terms = build_all_course_terms(sections_file=sections_file)

    dashboard = build_weekly_dashboard(course_terms, start, end)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / f"weekly_dashboard_{start.isoformat()}_to_{end.isoformat()}.md"
    output_file.write_text(dashboard, encoding="utf-8")

    print(f"Weekly dashboard created: {output_file}")
    return output_file


#if __name__ == "__main__":
    # Normal Friday use: generates next Monday-Sunday.
    generate_weekly_dashboard()

    # Useful test/planning calls:
    # generate_weekly_dashboard(start_date="2026-09-09", end_date="2026-09-13")  # Week 0
    # generate_weekly_dashboard(start_date="2026-09-14", end_date="2026-09-20")  # Week 1


In [3]:
generate_weekly_dashboard(start_date="2026-09-21")

MOS 2320
RETURN has_midterm: False
RETURN midterm: {'date': 'NA', 'time': 'NA', 'location': 'TBD'}
MOS 2320
RETURN has_midterm: False
RETURN midterm: {'date': 'NA', 'time': 'NA', 'location': 'TBD'}
MOS 3321
RETURN has_midterm: True
RETURN midterm: {'date': '2026-10-24', 'time': '2 - 4 PM', 'location': 'Kingsmill'}
MOS 3321
RETURN has_midterm: True
RETURN midterm: {'date': '2026-10-24', 'time': '2 - 4 PM', 'location': 'Kingsmill'}
Weekly dashboard created: outputs\weekly_briefs\weekly_dashboard_2026-09-21_to_2026-09-27.md


WindowsPath('outputs/weekly_briefs/weekly_dashboard_2026-09-21_to_2026-09-27.md')

In [ ]:
parse_meeting_string(
    "Mondays 8:30 AM - 10:30 AM & Wednesdays 8:30 AM – 9:30 AM"
)